# Grover Search with Strict Single-Shot Contextual Error Correction

This notebook demonstrates Grover search under a **strict single-shot measurement regime** while explicitly separating three different sources of displacement:

1. **Input-dependent legitimate displacement** — the marked item changes the oracle and therefore changes the correct trajectory.
2. **Circuit-dependent legitimate displacement** — oracle and diffusion operations intentionally move the state through Hilbert space during amplitude amplification.
3. **Context-propagated error displacement** — coherent errors are injected *inside* circuit depth and are subsequently transformed by all later Grover operations.

The clean trajectory is the contextual expected reference. Therefore, neither the information encoded by the marked item nor the legitimate evolution produced by Grover gates is classified as error.

For the controlled example,

$$
n=3,\qquad N=2^n=8,\qquad |w\rangle=|101\rangle.
$$

The approximate optimal number of Grover iterations is

$$
k_{\mathrm{opt}}=\left\lfloor\frac{\pi}{4}\sqrt{N}\right\rfloor=2.
$$

The notebook uses **one physical sample per trajectory** at the measurement layer:

$$
S=1.
$$

Exact statevectors are retained only as independently propagated **expected-value references** needed to audit the model. They are not treated as additional measurement shots.

At the observable-distribution level, let

$$
E^{\mathrm{ideal}},\qquad E^{\mathrm{noisy}},\qquad M^{(1)}
$$

denote the clean expected distribution, noisy expected distribution, and strict one-shot observation respectively. We decompose the total residual as

$$
R=M^{(1)}-E^{\mathrm{ideal}}
=\underbrace{\left(E^{\mathrm{noisy}}-E^{\mathrm{ideal}}\right)}_{\Delta^{\mathrm{hw}}}
+\underbrace{\left(M^{(1)}-E^{\mathrm{noisy}}\right)}_{S^{(1)}}.
$$

The deterministic reference projection is

$$
M^{\mathrm{corr}}=M^{(1)}-R=E^{\mathrm{ideal}}.
$$

Accordingly, exact reference recovery is an algebraic property of this controlled correction definition; it is **not** a claim that arbitrary unknown hardware noise can be inferred perfectly from one experimental shot.

## 1. Installing the required packages

This notebook uses only the packages required for numerical operations and quantum-circuit simulation.

- `PennyLane` provides the quantum gates, QNodes, and statevector simulation interface.
- `pennylane-lightning` provides a faster statevector backend when it is available.
- `NumPy` is used for linear algebra, probabilities, sampling, and error metrics.

Unlike the molecular-Hamiltonian example, Grover search does not require `PySCF`, `OpenFermion`, or `openfermionpyscf`.

The installation cell below is intended to be directly runnable in Google Colab.

In [ ]:
!pip install -q pennylane pennylane-lightning numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 9.2 MB/s eta 0:00:00


## 2. Experimental configuration

With three qubits, the search Hilbert space is

$$
\mathcal{H}=(\mathbb{C}^{2})^{\otimes 3},
$$

with dimension

$$
N=2^3=8.
$$

We begin in the computational basis state $|000\rangle$ and apply a Hadamard gate to every qubit. This creates the uniform superposition

$$
|s\rangle
=
H^{\otimes n}|0\rangle^{\otimes n}
=
\frac{1}{\sqrt{N}}
\sum_{x=0}^{N-1}|x\rangle.
$$

Therefore, before Grover amplification, every basis state has the same probability:

$$
P(x)=\frac{1}{N}=\frac{1}{8}.
$$

The marked state is chosen as $|101\rangle$. This target information is used to construct the oracle; it is not inserted into the final inference procedure.

In [ ]:
import numpy as np
import pennylane as qml

SEED = 42
NOISE_SEED = 314159
rng_noise = np.random.default_rng(NOISE_SEED)

N_QUBITS = 3
WIRES = list(range(N_QUBITS))
DIM = 2 ** N_QUBITS

TARGET_BITS = "101"
TARGET_INDEX = int(TARGET_BITS, 2)

N_ITERATIONS = int(np.floor((np.pi / 4.0) * np.sqrt(DIM)))
SHOTS = 1
EPS = 1e-12

try:
    dev = qml.device("lightning.qubit", wires=N_QUBITS)
    BACKEND = "lightning.qubit"
except Exception:
    dev = qml.device("default.qubit", wires=N_QUBITS)
    BACKEND = "default.qubit"

print("=" * 82)
print("GROVER CONFIGURATION")
print("=" * 82)
print("Qubits                :", N_QUBITS)
print("Search-space size     :", DIM)
print("Marked state          :", f"|{TARGET_BITS}>")
print("Marked-state index    :", TARGET_INDEX)
print("Grover iterations     :", N_ITERATIONS)
print("Backend               :", BACKEND)
print("Strict measurement shots:", SHOTS)

GROVER CONFIGURATION
Qubits                : 3
Search-space size     : 8
Marked state          : |101>
Marked-state index    : 5
Grover iterations     : 2
Backend               : lightning.qubit
Strict measurement shots: 1


## 3. Oracle and diffusion operators

A Grover iteration contains two essential transformations: the **oracle** and the **diffusion operator**.

### Oracle

The oracle changes only the phase of the marked state:

$$
O_w|x\rangle=(-1)^{f(x)}|x\rangle,
$$

where

$$
f(x)=
\begin{cases}
1, & x=w,\\
0, & x\neq w.
\end{cases}
$$

Thus,

$$
O_w|w\rangle=-|w\rangle,
$$

while all unmarked computational-basis states remain unchanged.

### Diffusion operator

After the oracle, Grover's diffusion operator performs a reflection about the uniform state:

$$
D=2|s\rangle\langle s|-I.
$$

One complete Grover iteration is therefore

$$
G=DO_w.
$$

Geometrically, repeated applications of $G$ rotate the state within the two-dimensional subspace spanned by the marked state and the normalized superposition of unmarked states:

$$
\mathrm{span}\{|w\rangle,|w^\perp\rangle\}.
$$

This rotation is **legitimate algorithmic evolution**. It must be preserved by the correction framework rather than being misclassified as an error.

In [ ]:
def apply_oracle():
    # Map |101> to |111>, then apply a multi-controlled Z.
    # TARGET_BITS = q0 q1 q2.
    for wire, bit in zip(WIRES, TARGET_BITS):
        if bit == "0":
            qml.PauliX(wires=wire)

    # CCZ = H(target) -> Toffoli -> H(target)
    qml.Hadamard(wires=WIRES[-1])
    qml.Toffoli(wires=WIRES)
    qml.Hadamard(wires=WIRES[-1])

    for wire, bit in zip(WIRES, TARGET_BITS):
        if bit == "0":
            qml.PauliX(wires=wire)


def apply_diffusion():
    # D = H^n X^n (2|000><000| - I) X^n H^n
    for wire in WIRES:
        qml.Hadamard(wires=wire)

    for wire in WIRES:
        qml.PauliX(wires=wire)

    qml.Hadamard(wires=WIRES[-1])
    qml.Toffoli(wires=WIRES)
    qml.Hadamard(wires=WIRES[-1])

    for wire in WIRES:
        qml.PauliX(wires=wire)

    for wire in WIRES:
        qml.Hadamard(wires=wire)

## 4. Contextual error model and propagation through circuit depth

The error model is inserted **inside** the Grover circuit rather than appended to its output. At contextual location $c$, each qubit receives a small coherent perturbation

$$
E_c=\prod_q R_X(\epsilon^X_{c,q})R_Y(\epsilon^Y_{c,q})R_Z(\epsilon^Z_{c,q}).
$$

If the legitimate circuit operation at depth $l$ is $U_l$, the clean expected trajectory obeys

$$
|\psi_l^{\mathrm{ideal}}\rangle=U_l|\psi_{l-1}^{\mathrm{ideal}}\rangle,
$$

whereas the noisy expected trajectory obeys

$$
|\psi_l^{\mathrm{noisy}}\rangle
=E_lU_l|\psi_{l-1}^{\mathrm{noisy}}\rangle.
$$

This recursion is essential. An error introduced early in the circuit becomes part of the input to every subsequent oracle and diffusion operation. Consequently, the final error is not merely the sum of independent local rotations; it is the result of **contextual propagation through the remaining algorithm**.

The clean and noisy expected trajectories are propagated independently through the same legitimate Grover structure. This allows the notebook to calculate

$$
\Delta_l^{\mathrm{hw}}
=E_l^{\mathrm{noisy}}-E_l^{\mathrm{ideal}}
$$

at each audited stage without confusing legitimate Grover evolution with error.

In [ ]:
RX_ERROR_RANGE = (-0.045, 0.045)
RY_ERROR_RANGE = (-0.055, 0.055)
RZ_ERROR_RANGE = (-0.045, 0.045)

# Contexts:
# 1) initial superposition
# 2) oracle after each Grover iteration
# 3) diffusion after each Grover iteration
N_CONTEXTS = 1 + 2 * N_ITERATIONS

ERROR_MAP = np.zeros((N_CONTEXTS, N_QUBITS, 3), dtype=np.float64)
ERROR_MAP[:, :, 0] = rng_noise.uniform(*RX_ERROR_RANGE, size=(N_CONTEXTS, N_QUBITS))
ERROR_MAP[:, :, 1] = rng_noise.uniform(*RY_ERROR_RANGE, size=(N_CONTEXTS, N_QUBITS))
ERROR_MAP[:, :, 2] = rng_noise.uniform(*RZ_ERROR_RANGE, size=(N_CONTEXTS, N_QUBITS))

def inject_contextual_error(context_index):
    for q in WIRES:
        ex, ey, ez = ERROR_MAP[context_index, q]
        qml.RX(ex, wires=q)
        qml.RY(ey, wires=q)
        qml.RZ(ez, wires=q)

print("=" * 82)
print("CONTEXTUAL ERROR MAP")
print("=" * 82)
print("Contexts              :", N_CONTEXTS)
print("Noise seed            :", NOISE_SEED)
print(f"RX observed range     : [{ERROR_MAP[:,:,0].min():+.5f}, {ERROR_MAP[:,:,0].max():+.5f}]")
print(f"RY observed range     : [{ERROR_MAP[:,:,1].min():+.5f}, {ERROR_MAP[:,:,1].max():+.5f}]")
print(f"RZ observed range     : [{ERROR_MAP[:,:,2].min():+.5f}, {ERROR_MAP[:,:,2].max():+.5f}]")

CONTEXTUAL ERROR MAP
Contexts              : 5
Noise seed            : 314159
RX observed range     : [-0.03581, +0.03704]
RY observed range     : [-0.04561, +0.03637]
RZ observed range     : [-0.03977, +0.04160]


## 5. Independent clean and noisy expected trajectories

To verify that legitimate circuit evolution is never interpreted as error, we audit the trajectory at five semantic checkpoints:

$$
|s\rangle
\rightarrow O_1
\rightarrow D_1
\rightarrow O_2
\rightarrow D_2.
$$

For each checkpoint $l$, two states are computed independently:

$$
|\psi_l^{\mathrm{ideal}}\rangle,
\qquad
|\psi_l^{\mathrm{noisy}}\rangle.
$$

The **legitimate circuit displacement** between consecutive clean checkpoints is

$$
D_l^{\mathrm{circuit}}
=|\psi_l^{\mathrm{ideal}}\rangle-|\psi_{l-1}^{\mathrm{ideal}}\rangle.
$$

The **context-propagated error displacement** is instead

$$
D_l^{\mathrm{error}}
=|\psi_l^{\mathrm{noisy}}\rangle-|\psi_l^{\mathrm{ideal}}\rangle.
$$

Because both trajectories use the same marked state and the same legitimate oracle/diffusion sequence, target-dependent and circuit-dependent changes remain inside the expected reference.

In [ ]:
def build_ideal_checkpoint_qnode(stage):
    @qml.qnode(dev)
    def circuit():
        for wire in WIRES:
            qml.Hadamard(wires=wire)

        if stage >= 1:
            apply_oracle()
        if stage >= 2:
            apply_diffusion()
        if stage >= 3:
            apply_oracle()
        if stage >= 4:
            apply_diffusion()
        return qml.state()
    return circuit


def build_noisy_checkpoint_qnode(stage):
    @qml.qnode(dev)
    def circuit():
        context = 0
        for wire in WIRES:
            qml.Hadamard(wires=wire)
        inject_contextual_error(context)
        context += 1

        if stage >= 1:
            apply_oracle()
            inject_contextual_error(context)
            context += 1
        if stage >= 2:
            apply_diffusion()
            inject_contextual_error(context)
            context += 1
        if stage >= 3:
            apply_oracle()
            inject_contextual_error(context)
            context += 1
        if stage >= 4:
            apply_diffusion()
            inject_contextual_error(context)
        return qml.state()
    return circuit

CHECKPOINT_NAMES = [
    "Uniform input",
    "After oracle 1",
    "After diffusion 1",
    "After oracle 2",
    "After diffusion 2",
]

ideal_states = []
noisy_states = []
for stage in range(5):
    ideal_states.append(np.asarray(build_ideal_checkpoint_qnode(stage)(), dtype=np.complex128))
    noisy_states.append(np.asarray(build_noisy_checkpoint_qnode(stage)(), dtype=np.complex128))

print("=" * 100)
print("INDEPENDENT EXPECTED-TRAJECTORY CHECKPOINTS")
print("=" * 100)
print("Clean and noisy references were propagated independently through every audited stage.")
print("The marked-item input and legitimate Grover gates are shared by both trajectories.")

INDEPENDENT EXPECTED-TRAJECTORY CHECKPOINTS
Clean and noisy references were propagated independently through every audited stage.
The marked-item input and legitimate Grover gates are shared by both trajectories.


## 6. Stage-by-stage expected-value audit

At each checkpoint we inspect the target-state probability

$$
E_l=P_l(w)=|\langle w|\psi_l\rangle|^2.
$$

Three quantities are printed:

**Legitimate circuit change**

$$
\Delta_l^{\mathrm{legit}}
=E_l^{\mathrm{ideal}}-E_{l-1}^{\mathrm{ideal}},
$$

which is caused by the intended Grover operation and must **not** be corrected.

**Context-propagated error**

$$
\Delta_l^{\mathrm{hw}}
=E_l^{\mathrm{noisy}}-E_l^{\mathrm{ideal}},
$$

which compares the noisy expected trajectory with the clean expected trajectory at the *same circuit depth*.

This same-depth comparison prevents legitimate data-dependent and circuit-dependent evolution from being mistaken for error.

In [ ]:
def normalize_state(state):
    state = np.asarray(state, dtype=np.complex128)
    norm = np.linalg.norm(state)
    if norm < EPS:
        raise ValueError("State norm is zero.")
    return state / norm


def align_global_phase(reference, state):
    overlap = np.vdot(reference, state)
    if abs(overlap) < EPS:
        return state
    phase = overlap / abs(overlap)
    return state * np.conj(phase)


def probs_from_state(state):
    state = normalize_state(state)
    p = np.abs(state) ** 2
    return p / p.sum()

ideal_states = [normalize_state(s) for s in ideal_states]
noisy_states = [align_global_phase(i, normalize_state(n)) for i, n in zip(ideal_states, noisy_states)]

ideal_probs = [probs_from_state(s) for s in ideal_states]
noisy_probs = [probs_from_state(s) for s in noisy_states]

print("=" * 100)
print("STAGE-BY-STAGE EXPECTED-VALUE / ERROR-PROPAGATION AUDIT")
print("=" * 100)
print(f"{'Checkpoint':<22} {'E_ideal(w)':>14} {'E_noisy(w)':>14} {'Legit Δ':>14} {'Error Δ':>14} {'State error RMS':>17}")
print("-" * 100)

previous_ideal_target = None
for k, name in enumerate(CHECKPOINT_NAMES):
    ei = float(ideal_probs[k][TARGET_INDEX])
    en = float(noisy_probs[k][TARGET_INDEX])
    legit = 0.0 if previous_ideal_target is None else ei - previous_ideal_target
    hw = en - ei
    state_err = np.sqrt(np.mean(np.abs(noisy_states[k] - ideal_states[k]) ** 2))
    print(f"{name:<22} {ei:14.8f} {en:14.8f} {legit:+14.8e} {hw:+14.8e} {state_err:17.8e}")
    previous_ideal_target = ei

print("-" * 100)
print("Legit Δ is expected algorithmic evolution; Error Δ is noisy-vs-ideal at identical depth.")

STAGE-BY-STAGE EXPECTED-VALUE / ERROR-PROPAGATION AUDIT
Checkpoint                 E_ideal(w)     E_noisy(w)        Legit Δ        Error Δ   State error RMS
----------------------------------------------------------------------------------------------------
Uniform input              0.12500000     0.11844450 +0.00000000e+00 -6.55550135e-03    8.83061081e-03
After oracle 1             0.12500000     0.12253316 +0.00000000e+00 -2.46683660e-03    1.42301074e-02
After diffusion 1          0.78125000     0.77145222 +6.56250000e-01 -9.79778002e-03    2.27038849e-02
After oracle 2             0.78125000     0.77207541 -1.11022302e-16 -9.17459097e-03    2.18724329e-02
After diffusion 2          0.94531250     0.94622916 +1.64062500e-01 +9.16664069e-04    2.77206086e-02
----------------------------------------------------------------------------------------------------
Legit Δ is expected algorithmic evolution; Error Δ is noisy-vs-ideal at identical depth.


## 7. Final contextual decomposition before single-shot measurement

At the final Grover depth, define

$$
|\psi_f^{\mathrm{ideal}}\rangle,
\qquad
|\psi_f^{\mathrm{noisy}}\rangle.
$$

The complete legitimate Grover displacement from the uniform input is

$$
D_{\mathrm{Grover}}
=|\psi_f^{\mathrm{ideal}}\rangle-|\psi_0^{\mathrm{ideal}}\rangle.
$$

The propagated contextual error is

$$
D_{\mathrm{error}}
=|\psi_f^{\mathrm{noisy}}\rangle-|\psi_f^{\mathrm{ideal}}\rangle.
$$

These quantities are deliberately defined against different references: the first measures intended algorithmic motion, whereas the second compares clean and noisy trajectories at the same final depth.

The expected corrected reference is obtained by deterministic residual projection:

$$
|\psi_f^{\mathrm{corr}}\rangle
=|\psi_f^{\mathrm{noisy}}\rangle-D_{\mathrm{error}}
=|\psi_f^{\mathrm{ideal}}\rangle.
$$

In [ ]:
uniform = ideal_states[0]
ideal = ideal_states[-1]
noisy = noisy_states[-1]

legitimate_grover_displacement = ideal - uniform
context_error = noisy - ideal
corrected_expected_state = normalize_state(noisy - context_error)
post_correction_residual = corrected_expected_state - ideal

grover_rms = np.sqrt(np.mean(np.abs(legitimate_grover_displacement) ** 2))
error_rms = np.sqrt(np.mean(np.abs(context_error) ** 2))
post_rms = np.sqrt(np.mean(np.abs(post_correction_residual) ** 2))

num = np.real(np.vdot(legitimate_grover_displacement, context_error))
den = np.linalg.norm(legitimate_grover_displacement) * np.linalg.norm(context_error) + EPS
alignment = float(num / den)

print("=" * 82)
print("FINAL GROVER / CONTEXTUAL ERROR DECOMPOSITION")
print("=" * 82)
print(f"Legitimate Grover displacement RMS : {grover_rms:.8e}")
print(f"Context-propagated error RMS       : {error_rms:.8e}")
print(f"Expected correction residual RMS   : {post_rms:.8e}")
print(f"Grover↔error real alignment        : {alignment:+.8f}")
print("Legitimate input/circuit evolution remains in the reference; only same-depth error is corrected.")

FINAL GROVER / CONTEXTUAL ERROR DECOMPOSITION
Legitimate Grover displacement RMS : 4.67707173e-01
Context-propagated error RMS       : 2.77206086e-02
Expected correction residual RMS   : 7.95706899e-17
Grover↔error real alignment        : -0.16736597
Legitimate input/circuit evolution remains in the reference; only same-depth error is corrected.


## 8. Strict single-shot measurement model

The experiment now enters the actual measurement layer. Unlike the previous version, **only one sample is drawn from each final trajectory**:

$$
S=1.
$$

The expected distributions

$$
E^{\mathrm{ideal}}=P_{\mathrm{ideal}},\qquad
E^{\mathrm{noisy}}=P_{\mathrm{noisy}}
$$

are model references obtained from independent state propagation. The one-shot noisy measurement is represented as a one-hot vector $M^{(1)}$.

For example, if the single measured bitstring is $101$, then

$$
M^{(1)}=(0,0,0,0,0,1,0,0).
$$

We then separate the total one-shot residual into

$$
\Delta^{\mathrm{hw}}
=E^{\mathrm{noisy}}-E^{\mathrm{ideal}}
$$

and

$$
S^{(1)}=M^{(1)}-E^{\mathrm{noisy}}.
$$

Thus

$$
R=M^{(1)}-E^{\mathrm{ideal}}
=\Delta^{\mathrm{hw}}+S^{(1)}.
$$

This explicitly distinguishes modeled contextual error from unavoidable single-shot sampling deviation.

## Information-leakage audit and correction boundary

The error injector and the correction stage are deliberately separated.

The **injection path** uses the hidden simulation-side contextual error map:

$$
\mathrm{ERROR\_MAP}
\longrightarrow
E_c
\longrightarrow
|\psi_{\mathrm{noisy}}\rangle.
$$

The **correction function itself** is restricted to only two inputs:

$$
M^{(1)}
\quad\text{and}\quad
E_{\mathrm{ideal}}.
$$

It does **not** receive the injected $R_X/R_Y/R_Z$ angles, `ERROR_MAP`, `NOISE_SEED`, the injection function, or the exact noisy expected distribution.

The exact noisy expectation $E_{\mathrm{noisy}}$ is retained only in an **audit-only branch** so that the final one-shot residual can be scientifically decomposed as

$$
R
=
M^{(1)}-E_{\mathrm{ideal}}
=
\underbrace{
E_{\mathrm{noisy}}-E_{\mathrm{ideal}}
}_{\Delta_{\mathrm{context}}}
+
\underbrace{
M^{(1)}-E_{\mathrm{noisy}}
}_{S^{(1)}}.
$$

This audit branch verifies how the injected error propagated through the circuit, but its value is not passed into the correction function.

The correction boundary is therefore

$$
R_{\mathrm{corr}}
=
M^{(1)}-E_{\mathrm{ideal}},
$$

$$
M_{\mathrm{corrected}}
=
M^{(1)}-R_{\mathrm{corr}}.
$$

This removes **direct injection-parameter leakage**. However, the method remains a **reference-based deterministic projection**, because the independently propagated ideal contextual expectation is intentionally available to the correction stage. It should therefore not be described as blind inference of an unknown hardware error from one shot alone.

The correction percentage is reported as

$$
\mathrm{Correction}\;(\%)
=
100
\left(
1-
\frac{
\mathrm{MAE}(M_{\mathrm{corrected}},E_{\mathrm{ideal}})
}{
\mathrm{MAE}(M^{(1)},E_{\mathrm{ideal}})
}
\right).
$$

Under the deterministic reference-projection definition, a value numerically equal to 100% is expected when the corrected vector exactly returns to the contextual ideal reference.

In [ ]:
P_uniform = ideal_probs[0]
P_ideal = ideal_probs[-1]
P_noisy = noisy_probs[-1]
P_corrected_expected = probs_from_state(corrected_expected_state)

# Strict one-shot sampling: exactly one draw per trajectory.
shot_rng = np.random.default_rng(SEED + 8080)

def one_shot_vector(probs):
    outcome = int(shot_rng.choice(len(probs), size=1, p=probs)[0])
    vec = np.zeros_like(probs, dtype=np.float64)
    vec[outcome] = 1.0
    return outcome, vec

ideal_outcome, M_ideal_1 = one_shot_vector(P_ideal)
noisy_outcome, M_noisy_1 = one_shot_vector(P_noisy)

# -------------------------------------------------------------------------
# AUDIT-ONLY DECOMPOSITION
# -------------------------------------------------------------------------
# P_noisy is used below ONLY to decompose the observed residual into:
#   (1) contextual/hardware expected displacement
#   (2) one-shot sampling displacement
#
# It is NOT passed into the correction function.
hardware_error_expected = P_noisy - P_ideal
single_shot_residual = M_noisy_1 - P_noisy
total_residual = M_noisy_1 - P_ideal

closure = total_residual - (
    hardware_error_expected + single_shot_residual
)

# -------------------------------------------------------------------------
# CORRECTION BOUNDARY
# -------------------------------------------------------------------------
# The correction function receives ONLY:
#   - the observed one-shot vector M(1)
#   - the independently propagated ideal contextual reference E_ideal
#
# It receives NO ERROR_MAP, injected RX/RY/RZ angles, NOISE_SEED,
# inject_contextual_error(), or P_noisy.
def contextual_reference_correction(measured_one_shot, ideal_expected):
    residual = measured_one_shot - ideal_expected
    corrected = measured_one_shot - residual
    return corrected, residual

M_corrected, correction_residual = contextual_reference_correction(
    M_noisy_1,
    P_ideal
)

# Correction percentage:
# compare distance to the ideal contextual reference before and after correction.
pre_correction_mae = np.mean(np.abs(M_noisy_1 - P_ideal))
post_correction_mae = np.mean(np.abs(M_corrected - P_ideal))

if pre_correction_mae > EPS:
    correction_percentage = 100.0 * (
        1.0 - post_correction_mae / pre_correction_mae
    )
else:
    correction_percentage = 100.0 if post_correction_mae <= EPS else 0.0

print("=" * 96)
print("STRICT SINGLE-SHOT RESIDUAL DECOMPOSITION")
print("=" * 96)
print(f"Shots per measured trajectory        : {SHOTS}")
print(f"Ideal one-shot outcome               : |{format(ideal_outcome, f'0{N_QUBITS}b')}>")
print(f"Noisy one-shot outcome               : |{format(noisy_outcome, f'0{N_QUBITS}b')}>")
print(f"Hardware/context expected MAE        : {np.mean(np.abs(hardware_error_expected)):.8e}")
print(f"Single-shot sampling residual MAE    : {np.mean(np.abs(single_shot_residual)):.8e}")
print(f"Total one-shot residual MAE          : {np.mean(np.abs(total_residual)):.8e}")
print(f"Residual decomposition closure max Δ : {np.max(np.abs(closure)):.8e}")
print(f"Pre-correction → ideal MAE           : {pre_correction_mae:.8e}")
print(f"Post-correction → ideal MAE          : {post_correction_mae:.8e}")
print(f"ERROR CORRECTION PERCENTAGE          : {correction_percentage:.6f}%")

STRICT SINGLE-SHOT RESIDUAL DECOMPOSITION
Shots per measured trajectory        : 1
Ideal one-shot outcome               : |101>
Noisy one-shot outcome               : |101>
Hardware/context expected MAE        : 2.16810655e-03
Single-shot sampling residual MAE    : 1.34427090e-02
Total one-shot residual MAE          : 1.36718750e-02
Residual decomposition closure max Δ : 0.00000000e+00
Pre-correction → ideal MAE           : 1.36718750e-02
Post-correction → ideal MAE          : 0.00000000e+00
ERROR CORRECTION PERCENTAGE          : 100.000000%


## 9. Per-state expected values, one-shot observation, and correction

The table below makes the correction mechanism explicit for every computational-basis state $x$.

For each state we print:

$$
E_x^{\mathrm{ideal}},\quad
E_x^{\mathrm{noisy}},\quad
M_x^{(1)},\quad
\Delta_x^{\mathrm{hw}},\quad
S_x^{(1)},\quad
R_x,\quad
M_x^{\mathrm{corr}}.
$$

The identity checked numerically is

$$
R_x=\Delta_x^{\mathrm{hw}}+S_x^{(1)}
$$

and therefore

$$
M_x^{\mathrm{corr}}=M_x^{(1)}-R_x=E_x^{\mathrm{ideal}}.
$$

This table is also useful for verifying that the expected value associated with the marked state is amplified by legitimate Grover evolution before the error term is evaluated.

In [ ]:
print("=" * 132)
print("PER-STATE EXPECTED VALUE / SINGLE-SHOT / CORRECTION AUDIT")
print("=" * 132)
print(f"{'state':<8} {'E_ideal':>11} {'E_noisy':>11} {'M(1)':>8} {'Δ_hw':>12} {'S(1)':>12} {'R_total':>12} {'M_corr':>11}")
print("-" * 132)
for idx in range(DIM):
    label = format(idx, f"0{N_QUBITS}b")
    print(
        f"|{label}>  "
        f"{P_ideal[idx]:11.7f} {P_noisy[idx]:11.7f} {M_noisy_1[idx]:8.1f} "
        f"{hardware_error_expected[idx]:+12.5e} {single_shot_residual[idx]:+12.5e} "
        f"{total_residual[idx]:+12.5e} {M_corrected[idx]:11.7f}"
    )

PER-STATE EXPECTED VALUE / SINGLE-SHOT / CORRECTION AUDIT
state        E_ideal     E_noisy     M(1)         Δ_hw         S(1)      R_total      M_corr
------------------------------------------------------------------------------------------------------------------------------------
|000>    0.0078125   0.0063772      0.0 -1.43534e-03 -6.37716e-03 -7.81250e-03   0.0078125
|001>    0.0078125   0.0146667      0.0 +6.85417e-03 -1.46667e-02 -7.81250e-03   0.0078125
|010>    0.0078125   0.0054071      0.0 -2.40545e-03 -5.40705e-03 -7.81250e-03   0.0078125
|011>    0.0078125   0.0063831      0.0 -1.42936e-03 -6.38314e-03 -7.81250e-03   0.0078125
|100>    0.0078125   0.0086033      0.0 +7.90831e-04 -8.60333e-03 -7.81250e-03   0.0078125
|101>    0.9453125   0.9462292      1.0 +9.16664e-04 +5.37708e-02 +5.46875e-02   0.9453125
|110>    0.0078125   0.0079233      0.0 +1.10762e-04 -7.92326e-03 -7.81250e-03   0.0078125
|111>    0.0078125   0.0044102      0.0 -3.40227e-03 -4.41023e-03 -7.81250e-03 

## 10. Target-state amplification and blind inference

The marked state begins with uniform probability

$$
P_0(w)=\frac{1}{8}.
$$

Grover's intended dynamics increase this expected probability. That increase is legitimate algorithmic signal, not error.

We therefore compare

$$
P_{\mathrm{uniform}}(w),\quad
E_{\mathrm{ideal}}(w),\quad
E_{\mathrm{noisy}}(w),\quad
M^{\mathrm{corr}}(w).
$$

For algorithmic inference, the expected corrected distribution is evaluated blindly using

$$
\hat{x}=\operatorname*{arg\,max}_x M^{\mathrm{corr}}_x.
$$

The inference routine receives no target label.

In [ ]:
p0 = float(P_uniform[TARGET_INDEX])
pi = float(P_ideal[TARGET_INDEX])
pn = float(P_noisy[TARGET_INDEX])
pc = float(M_corrected[TARGET_INDEX])

amplification = pi / (p0 + EPS)


def infer_search_result(probs):
    idx = int(np.argmax(probs))
    return idx, format(idx, f"0{N_QUBITS}b"), float(probs[idx])

ideal_result = infer_search_result(P_ideal)
noisy_expected_result = infer_search_result(P_noisy)
corrected_result = infer_search_result(M_corrected)

print("=" * 82)
print("TARGET AMPLIFICATION + BLIND INFERENCE")
print("=" * 82)
print(f"Target state                       : |{TARGET_BITS}>")
print(f"Uniform target expected value      : {p0:.8f}")
print(f"Ideal Grover expected value        : {pi:.8f}")
print(f"Noisy Grover expected value        : {pn:.8f}")
print(f"Corrected target expected value    : {pc:.8f}")
print(f"Legitimate amplification factor    : {amplification:.6f}x")
print(f"Noisy single-shot observed state   : |{format(noisy_outcome, f'0{N_QUBITS}b')}>")
print(f"Ideal expected blind result        : |{ideal_result[1]}>")
print(f"Noisy expected blind result        : |{noisy_expected_result[1]}>")
print(f"Corrected blind result             : |{corrected_result[1]}>")

TARGET AMPLIFICATION + BLIND INFERENCE
Target state                       : |101>
Uniform target expected value      : 0.12500000
Ideal Grover expected value        : 0.94531250
Noisy Grover expected value        : 0.94622916
Corrected target expected value    : 0.94531250
Legitimate amplification factor    : 7.562500x
Noisy single-shot observed state   : |101>
Ideal expected blind result        : |101>
Noisy expected blind result        : |101>
Corrected blind result             : |101>


## 11. Interpretation of a strict one-shot experiment

A single shot does **not** provide enough information to reconstruct an unknown probability distribution experimentally. This notebook therefore keeps the roles of measurement and reference modeling explicit.

The strict one-shot quantity is the observed vector

$$
M^{(1)}.
$$

The quantities

$$
E^{\mathrm{ideal}},\qquad E^{\mathrm{noisy}}
$$

are independently propagated expected references supplied by the controlled simulation model. They are required to decompose

$$
M^{(1)}-E^{\mathrm{ideal}}
$$

into contextual error and sampling deviation.

Therefore, this experiment demonstrates that the proposed deterministic correction rule can be **evaluated with a one-shot observation when the contextual expected references are available**. It does not claim that those complete expected distributions can themselves be learned from one shot on unknown hardware.

In [ ]:
print("=" * 82)
print("SINGLE-SHOT SCIENTIFIC CONSISTENCY CHECK")
print("=" * 82)
print(f"Configured shots                         : {SHOTS}")
print(f"One-hot noisy measurement sum            : {M_noisy_1.sum():.1f}")
print(f"Ideal expected distribution sum          : {P_ideal.sum():.12f}")
print(f"Noisy expected distribution sum          : {P_noisy.sum():.12f}")
print(f"Corrected reference distribution sum      : {M_corrected.sum():.12f}")
print(f"Decomposition closure max absolute error  : {np.max(np.abs(closure)):.8e}")
print(f"Correction reference max absolute error   : {np.max(np.abs(M_corrected-P_ideal)):.8e}")

print()
print("CORRECTION INPUT BOUNDARY")
print("-" * 82)
print("Correction inputs                       : M_noisy_1, P_ideal")
print("Injection parameters passed to correction: NONE")
print("P_noisy passed to correction             : NO (audit-only)")
print(f"ERROR CORRECTION PERCENTAGE               : {correction_percentage:.6f}%")


SINGLE-SHOT SCIENTIFIC CONSISTENCY CHECK
Configured shots                         : 1
One-hot noisy measurement sum            : 1.0
Ideal expected distribution sum          : 1.000000000000
Noisy expected distribution sum          : 1.000000000000
Corrected reference distribution sum      : 1.000000000000
Decomposition closure max absolute error  : 0.00000000e+00
Correction reference max absolute error   : 0.00000000e+00

CORRECTION INPUT BOUNDARY
----------------------------------------------------------------------------------
Correction inputs                       : M_noisy_1, P_ideal
Injection parameters passed to correction: NONE
P_noisy passed to correction             : NO (audit-only)
ERROR CORRECTION PERCENTAGE               : 100.000000%


## 12. Final scientific audit

The notebook now enforces the following separation.

### A. Legitimate input-dependent evolution

The marked item $|w\rangle$ determines the oracle. Its effect is part of the clean expected trajectory and is never defined as error.

### B. Legitimate circuit-dependent evolution

Oracle and diffusion operations intentionally transform the state:

$$
|s\rangle\rightarrow O_1\rightarrow D_1\rightarrow O_2\rightarrow D_2.
$$

At every stage, this intended motion is measured from the clean trajectory itself and remains outside the error term.

### C. Context-propagated error

Errors are injected between legitimate operations and therefore propagate through all later circuit transformations. At equal circuit depth,

$$
\Delta_l^{\mathrm{hw}}
=E_l^{\mathrm{noisy}}-E_l^{\mathrm{ideal}}.
$$

### D. Strict one-shot sampling

Only one final noisy outcome is drawn:

$$
S=1.
$$

Its sampling residual is

$$
S^{(1)}=M^{(1)}-E^{\mathrm{noisy}}.
$$

The complete residual satisfies

$$
M^{(1)}-E^{\mathrm{ideal}}
=\Delta^{\mathrm{hw}}+S^{(1)}.
$$

### E. Deterministic reference projection

Finally,

$$
M^{\mathrm{corr}}
=M^{(1)}-\left(M^{(1)}-E^{\mathrm{ideal}}\right)
=E^{\mathrm{ideal}}.
$$

Thus exact reference recovery is expected by construction. The scientifically relevant audit is that the expected reference evolves with the data and the circuit, while the injected error is propagated independently and compared only at matching circuit depth.

In [ ]:
print("=" * 88)
print("FINAL GROVER + STRICT SINGLE-SHOT CONTEXTUAL CORRECTION SUMMARY")
print("=" * 88)
print(f"Search space                         : {DIM} states")
print(f"Marked state                         : |{TARGET_BITS}>")
print(f"Grover iterations                    : {N_ITERATIONS}")
print(f"Measurement shots                    : {SHOTS}")
print(f"Legitimate Grover displacement RMS   : {grover_rms:.8e}")
print(f"Context-propagated error RMS         : {error_rms:.8e}")
print(f"Expected correction residual RMS     : {post_rms:.8e}")
print(f"Grover↔error alignment               : {alignment:+.8f}")
print(f"Ideal target expected value          : {pi:.8f}")
print(f"Noisy target expected value          : {pn:.8f}")
print(f"Noisy one-shot outcome               : |{format(noisy_outcome, f'0{N_QUBITS}b')}>")
print(f"Single-shot residual MAE             : {np.mean(np.abs(single_shot_residual)):.8e}")
print(f"Context/hardware expected MAE        : {np.mean(np.abs(hardware_error_expected)):.8e}")
print(f"Decomposition closure max Δ          : {np.max(np.abs(closure)):.8e}")
print(f"Corrected → ideal reference MAE      : {np.mean(np.abs(M_corrected-P_ideal)):.8e}")
print(f"ERROR CORRECTION PERCENTAGE          : {correction_percentage:.6f}%")
print(f"Corrected blind result               : |{corrected_result[1]}>")
print("=" * 88)

FINAL GROVER + STRICT SINGLE-SHOT CONTEXTUAL CORRECTION SUMMARY
Search space                         : 8 states
Marked state                         : |101>
Grover iterations                    : 2
Measurement shots                    : 1
Legitimate Grover displacement RMS   : 4.67707173e-01
Context-propagated error RMS         : 2.77206086e-02
Expected correction residual RMS     : 7.95706899e-17
Grover↔error alignment               : -0.16736597
Ideal target expected value          : 0.94531250
Noisy target expected value          : 0.94622916
Noisy one-shot outcome               : |101>
Single-shot residual MAE             : 1.34427090e-02
Context/hardware expected MAE        : 2.16810655e-03
Decomposition closure max Δ          : 0.00000000e+00
Corrected → ideal reference MAE      : 0.00000000e+00
ERROR CORRECTION PERCENTAGE          : 100.000000%
Corrected blind result               : |101>
